In [1]:
%pip install -q dotenv llama_stack_client==0.3.5 fire

Note: you may need to restart the kernel to use updated packages.


In [2]:
import uuid

import requests
from io import BytesIO

from llama_stack_client import Agent
from llama_stack_client import AgentEventLogger
from llama_stack_client import LlamaStackClient

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

base_url = os.getenv("REMOTE_BASE_URL")

client = LlamaStackClient(
    base_url=base_url
)

models = client.models.list()
models

INFO:httpx:HTTP Request: GET http://llamastack-distribution-vllm-service:8321/v1/models "HTTP/1.1 200 OK"


[Model(identifier='sentence-transformers/ibm-granite/granite-embedding-125m-english', metadata={'embedding_dimension': 768.0}, api_model_type='embedding', provider_id='sentence-transformers', type='model', provider_resource_id='ibm-granite/granite-embedding-125m-english', model_type='embedding'),
 Model(identifier='sentence-transformers/nomic-ai/nomic-embed-text-v1.5', metadata={'embedding_dimension': 768.0}, api_model_type='embedding', provider_id='sentence-transformers', type='model', provider_resource_id='nomic-ai/nomic-embed-text-v1.5', model_type='embedding'),
 Model(identifier='llama-guard/llama-guard-3-1b', metadata={}, api_model_type='llm', provider_id='llama-guard', type='model', provider_resource_id='llama-guard-3-1b', model_type='llm'),
 Model(identifier='vllm/qwen3-8b', metadata={}, api_model_type='llm', provider_id='vllm', type='model', provider_resource_id='qwen3-8b', model_type='llm')]

In [4]:
embedding_model=os.getenv("VDB_EMBEDDING", "sentence-transformers/ibm-granite/granite-embedding-125m-english")
embedding_dimension=int(os.getenv("VDB_EMBEDDING_DIMENSION", 384))

vs = client.vector_stores.create(
    name="hr-benefits-hybrid",
    extra_body={
        "embedding_model": embedding_model,
        "embedding_dimension": embedding_dimension,
        "search_mode": "hybrid",  # Enable hybrid search (keyword + semantic)
        "bm25_weight": 0.5,  # Weight for keyword search (BM25)
        "semantic_weight": 0.5,  # Weight for semantic search
    }
)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-vllm-service:8321/v1/vector_stores "HTTP/1.1 200 OK"


In [5]:
url = "https://raw.githubusercontent.com/burrsutter/fantaco-redhat-one-2026/refs/heads/main/rag-llama-stack/source_docs/FantaCoFabulousHRBenefits_clean.txt"

response = requests.get(url, timeout=30)
text_content = response.text

In [6]:
text_buffer = BytesIO(text_content.encode('utf-8'))
text_buffer.name = "hr-benefits-clean.txt"

uploaded_file = client.files.create(
    file=text_buffer,
    purpose="assistants"
)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-vllm-service:8321/v1/files "HTTP/1.1 200 OK"


In [7]:
client.vector_stores.files.create(
    vector_store_id=vs.id,
    file_id=uploaded_file.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 100,
            "chunk_overlap_tokens": 10
        }
    }
)

INFO:httpx:HTTP Request: POST http://llamastack-distribution-vllm-service:8321/v1/vector_stores/vs_41a872f7-11de-4586-a91e-09900afd48ea/files "HTTP/1.1 200 OK"


VectorStoreFile(id='file-53d0ae56aa3c4d3a9c9e129befd3ca74', attributes={}, chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(chunk_overlap_tokens=10, max_chunk_size_tokens=100), type='static'), created_at=1771397926, object='vector_store.file', status='completed', usage_bytes=0, vector_store_id='vs_41a872f7-11de-4586-a91e-09900afd48ea', last_error=None)

In [8]:
model = "vllm/qwen3-8b"
agent = Agent(
    client,
    model=model,
    instructions="You MUST use the file_search tool to answer ALL questions by searching the provided documents.",
    tools=[
        {
            "type": "file_search",
            "vector_store_ids": [vs.id],
        }
    ],
)

In [9]:
query = "What do I receive when I retire?"
session_id = agent.create_session("retirement-benefits-query")
response = agent.create_turn(
    messages=[{"role": "user", "content": query}],
    session_id=session_id,
    stream=True,
)

# Stream the response
for log in AgentEventLogger().log(response):
    print(log, end="")

INFO:httpx:HTTP Request: POST http://llamastack-distribution-vllm-service:8321/v1/conversations "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://llamastack-distribution-vllm-service:8321/v1/responses "HTTP/1.1 200 OK"


🤔 <think>
Okay, the user is asking, "What do I receive when I retire?" I need to figure out what information they need. Retiring can mean different things depending on their country, employment type, and benefits. Since they didn't specify, I should consider common retirement benefits.

First, I should check if there's specific information in the provided documents. The user mentioned using the file_search tool, so I should call the knowledge_search function with relevant queries. 

Possible areas to cover: pension benefits, social security, retirement savings plans like 401(k) or IRA, health insurance, and maybe other perks. But since the user might be in a specific country, like the US, I should include general terms. 

I'll start by searching for "retirement benefits" to get a broad overview. Then, maybe break down into specific areas like pension, social security, and savings. That way, the search terms cover different aspects they might be interested in.
</think>





🔧 Executing 